In [10]:
import os
import torch
import tifffile
from torch.utils.data import Dataset

class SentinelDataset(Dataset):

    def __init__(self, root_dir):

        self.samples = []

        classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])

        self.class_to_idx = {
            cls: idx
            for idx, cls in enumerate(classes)
        }

        print("Classes:", classes)

        for cls in classes:

            cls_path = os.path.join(root_dir, cls)

            count = 0

            for file in os.listdir(cls_path):

                if file.endswith(".tif"):

                    self.samples.append(
                        (
                            os.path.join(cls_path, file),
                            self.class_to_idx[cls]
                        )
                    )

                    count += 1

            print(f"{cls}: {count}")

        print("Total Samples:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        path, label = self.samples[idx]

        # print("Loading:", path)

        img = tifffile.imread(path)

        # print("Loaded")

        img = torch.tensor(
            img,
            dtype=torch.float32
        )

        img = img.permute(2,0,1)

        return img, label

In [11]:
from torch.utils.data import DataLoader

train_dataset = SentinelDataset(
    "EuroSAT_Dataset/EuroSATallBands/train"
)

val_dataset = SentinelDataset(
    "EuroSAT_Dataset/EuroSATallBands/val"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
AnnualCrop: 2100
Forest: 2100
HerbaceousVegetation: 2100
Highway: 1750
Industrial: 1750
Pasture: 1400
PermanentCrop: 1750
Residential: 2100
River: 1750
SeaLake: 2100
Total Samples: 18900
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
AnnualCrop: 450
Forest: 450
HerbaceousVegetation: 450
Highway: 375
Industrial: 375
Pasture: 300
PermanentCrop: 375
Residential: 450
River: 375
SeaLake: 450
Total Samples: 4050


In [12]:
import timm
import torch.nn as nn

NUM_CLASSES = 10
IN_CHANNELS = 13

model = timm.create_model(
    "convnext_tiny",
    pretrained=False,
    num_classes=NUM_CLASSES,
    in_chans=IN_CHANNELS
)

In [13]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

model = model.to(device)

Device: cuda
NVIDIA GeForce RTX 4050 Laptop GPU


In [14]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)

In [15]:
from tqdm.auto import tqdm
import torch

scaler = torch.amp.GradScaler()

def train_one_epoch():

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(
        train_loader,
        desc="Training",
        leave=True
    )

    for images, labels in pbar:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()

        total += labels.size(0)

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{correct/total:.4f}"
        })

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [16]:
@torch.no_grad()
def validate():

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(
        val_loader,
        desc="Validation",
        leave=True
    )

    for images, labels in pbar:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        running_loss += loss.item()

        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()

        total += labels.size(0)

        pbar.set_postfix({
            "acc": f"{correct/total:.4f}"
        })

    val_loss = running_loss / len(val_loader)
    val_acc = correct / total

    return val_loss, val_acc

In [17]:
x, y = train_dataset[0]

print(x.shape)
print(x.dtype)
print(y)

torch.Size([13, 64, 64])
torch.float32
0


In [18]:
EPOCHS = 30

best_acc = 0.0

for epoch in range(EPOCHS):

    print(
        f"\n{'='*20} "
        f"Epoch {epoch+1}/{EPOCHS} "
        f"{'='*20}"
    )

    train_loss, train_acc = train_one_epoch()

    val_loss, val_acc = validate()

    scheduler.step()

    print(
        f"\nTrain Loss : {train_loss:.4f}"
    )
    print(
        f"Train Acc  : {train_acc:.4f}"
    )
    print(
        f"Val Loss   : {val_loss:.4f}"
    )
    print(
        f"Val Acc    : {val_acc:.4f}"
    )

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_convnext.pth"
        )

        print(
            f"✓ Saved Best Model "
            f"(Val Acc = {best_acc:.4f})"
        )

print(
    f"\nBest Validation Accuracy: "
    f"{best_acc:.4f}"
)


==================== Epoch 1/30 ====================


Training:   0%|          | 0/296 [00:00<?, ?it/s]

Validation:   0%|          | 0/64 [00:00<?, ?it/s]


Train Loss : 1.1002
Train Acc  : 0.5988
Val Loss   : 0.7107
Val Acc    : 0.7560
✓ Saved Best Model (Val Acc = 0.7560)

==================== Epoch 2/30 ====================


Training:   0%|          | 0/296 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/64 [00:00<?, ?it/s]


Train Loss : 0.0006
Train Acc  : 1.0000
Val Loss   : 0.4574
Val Acc    : 0.8983

==================== Epoch 30/30 ====================


Training:   0%|          | 0/296 [00:00<?, ?it/s]

Validation:   0%|          | 0/64 [00:00<?, ?it/s]


Train Loss : 0.0006
Train Acc  : 1.0000
Val Loss   : 0.4575
Val Acc    : 0.8983

Best Validation Accuracy: 0.8985
